# SCG_ORIGINAL_REPRO_V1 — 실행 노트북

IBK투자증권 이정빈, 「바텀업 퀀트 – 알파 포트폴리오 전략」(2020-08-20)의
**스마트 컨센서스 갭 / 서프라이즈 포트폴리오** 재현.

> 이 노트북은 **런처와 조회 용도**다 (명세 §24). 핵심 로직은 전부 `smart_consensus_gap/`
> 패키지에 있고, 노트북에는 수천 줄을 박지 않는다.

## 실행되는 것

```
0 탐색(캐시 우선) → 1 정규화 → 2 PIT 스냅샷 → 3 애널리스트 정확도
→ 4·5 컨센서스/팩터 → 6 백테스트 → 7 강건성 → 8 감사·검증 → outputs/
```

## 데이터가 없으면

그럴듯한 숫자를 지어내지 않는다. 필수 테이블이 없으면 **합성 픽스처 리허설**로 전환해
계산경로와 감사만 증명하고, 모든 산출물에 `synthetic=true` 를 박는다.
실데이터 없이 나온 성과 수치는 성과가 아니다.

In [ ]:
!pip -q install numpy pandas pyarrow

## 실행 (한 셀)

In [ ]:
from run_smart_consensus_gap import run

result = run()

## 캐시/데이터 위치를 직접 지정할 때

탐색은 `SCG_DATA_DIR`, `./data`, `./scg_cache`, 구글드라이브 캐시
(`/content/drive/MyDrive/scg_cache`, 기존 `tcd_cache/_shared/table`)를 자동으로 훑는다.
그래도 못 찾으면 경로를 직접 준다.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# result = run(data_roots=['/content/drive/MyDrive/scg_cache'])
#
# 실데이터가 없으면 합성으로 넘어가지 말고 그냥 실패하게 하려면:
# result = run(allow_synthetic_fallback=False)

## 결과 조회 — 성과보다 감사와 재현도를 먼저 본다

In [ ]:
import pandas as pd

print('status =', result['status'], '/ mode =', result['mode'])
audit = result['audit']
display(audit[audit['severity'] == 'CRITICAL'][['check', 'status', 'detail']])

In [ ]:
display(result['performance'])
display(result['ic'])
display(result['quintiles'])
display(result['ablation'])
display(result['robustness'])

## 해석 문서 (§23)

In [ ]:
from IPython.display import Markdown

Markdown(result['interpretation'])

## 특정 종목의 스마트 컨센서스 역산 (§5.5, §28)

성분 감사표에는 estimator 단위의 원추정치·편향보정치·나이·최신성가중·skill·최종가중이
전부 남아 있다. `Σ(w_final × estimate_adj)` 가 그 시점의 smart consensus 와 일치해야 한다.

In [ ]:
import os

comp_path = os.path.join(result['out_dir'], '04_smart_consensus_components.parquet')
comp = pd.read_parquet(comp_path)

key = comp[['asof', 'company_code', 'fiscal_period']].iloc[0]
one = comp[(comp['asof'] == key['asof'])
           & (comp['company_code'] == key['company_code'])
           & (comp['fiscal_period'] == key['fiscal_period'])]
display(one[['estimator_id', 'published_at', 'age_days', 'estimate_raw', 'bias',
             'estimate_adj', 'w_recency', 'skill', 'w_final']])
print('Σw =', one['w_final'].sum())
print('smart consensus 역산 =', (one['w_final'] * one['estimate_adj']).sum())

## 2020-06-30 원문 표7 대조 (§14)

`smart_consensus_gap/reference/reference_20200630.csv` 는 비어 있다.
원문 PDF 16쪽 표7의 30종목을 직접 채워 넣으면 다음 실행부터 자동으로 대조된다.
**겹침이 낮다고 파라미터를 표7에 맞추는 것은 금지다(§14, §27).**

In [ ]:
display(result['reference'].T)